# leaf-tensor-condition — worked example 2: Identify leaf vs interior tensors after running a small forward graph

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `leaf-tensor-condition`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

After building a small computational graph with `wrap_forward_fn`, the input parameters are leaves (no recipe, requires_grad=True) while intermediate and output tensors are interior (they have a Recipe recording how they were computed). Only leaf tensors can accumulate `.grad` at backward time.

## Worked solution

Step 1: Build two leaf MiniTensors `x` and `y` with `requires_grad=True`.

Step 2: Use `wrap_forward_fn(np.multiply)` to compute `z = x * y` and `wrap_forward_fn(np.exp)` to compute `out = exp(z)`. Both `z` and `out` will have recipes; `x` and `y` will not.

Step 3: Call `is_leaf` on all four tensors and check: `x → True`, `y → True`, `z → False`, `out → False`.

Step 4: Print recipes to show that interior tensors carry them and leaf tensors do not.

In [ ]:
import numpy as np
from dataclasses import dataclass
from typing import Any, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Any
    args: tuple
    kwargs: dict
    parents: dict

class MiniTensor:
    def __init__(self, array, requires_grad=False):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe: Optional[Recipe] = None

def is_leaf(tensor: MiniTensor) -> bool:
    return tensor.recipe is None and tensor.requires_grad

def wrap_forward_fn(fwd_fn, is_differentiable=True):
    def tensor_func(*args, **kwargs):
        raw_args = tuple(a.array if isinstance(a, MiniTensor) else a for a in args)
        out_arr = fwd_fn(*raw_args, **kwargs)
        any_tracked = any(isinstance(a, MiniTensor) and a.requires_grad for a in args)
        rg = grad_tracking_enabled and is_differentiable and any_tracked
        out = MiniTensor(out_arr, rg)
        if rg:
            parents = {i: a for i, a in enumerate(args) if isinstance(a, MiniTensor)}
            out.recipe = Recipe(fwd_fn, raw_args, kwargs, parents)
        return out
    return tensor_func

wrapped_mul = wrap_forward_fn(np.multiply)
wrapped_exp = wrap_forward_fn(np.exp)

x = MiniTensor(np.array([2.0, 3.0]), requires_grad=True)
y = MiniTensor(np.array([0.5, 1.5]), requires_grad=True)

z   = wrapped_mul(x, y)
out = wrapped_exp(z)

for name, tensor in [('x', x), ('y', y), ('z', z), ('out', out)]:
    print(f'{name}: is_leaf={is_leaf(tensor)}, has_recipe={tensor.recipe is not None}')

assert is_leaf(x) == True
assert is_leaf(y) == True
assert is_leaf(z) == False
assert is_leaf(out) == False
print('Graph leaf classification correct.')